In [ ]:
import os
if os.path.exists('/content/checkmaize'):
    os.system('cd /content/checkmaize && git pull')
else:
    print("OPTION A: push this repo to GitHub and run: !git clone <your-repo-url> /content/checkmaize")
    print("OPTION B (no GitHub): zip the local repo (excluding node_modules, .git, artifacts) and upload to /content/checkmaize.zip, then:")
    print("  !mkdir -p /content/checkmaize && !unzip -q /content/checkmaize.zip -d /content/checkmaize")
print("Then run: !pip install -q -r /content/checkmaize/requirements.txt")


In [ ]:
import os
os.chdir('/content/checkmaize')
from datasets import load_dataset
from PIL import Image
import shutil

dst = 'data/raw/plantvillage'
shutil.rmtree(dst, ignore_errors=True)
CLASS_MAP = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_': 'Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight': 'Northern_Leaf_Blight',
    'Corn_(maize)___healthy': 'healthy',
}
os.makedirs(dst, exist_ok=True)
for split in ['train', 'test']:
    ds = load_dataset('mohanty/PlantVillage', split=split)
    corn = [r for r in ds if r['crop'] == 'Corn (maize)']
    print(f"{split}: {len(corn)} corn rows")
    for r in corn:
        folder = CLASS_MAP[r['label']]
        out = os.path.join(dst, folder, f"{r['leaf_id']}__{r['image_path'].rsplit('/', 1)[-1]}")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        r['image'].save(out)
print("plantvillage extraction done")


In [ ]:
import os, zipfile, shutil, glob
os.chdir('/content/checkmaize')

NEEDED = {'Leaf blight': 'Leaf blight', 'Leaf spot': 'Leaf spot', 'Healthy': 'Healthy'}
NEEDED_LOWER = {k.lower(): k for k in NEEDED}

candidates = []
for name in ['Raw Data.zip', 'crop-pest-and-disease-detection.zip']:
    p = f'/content/{name}'
    if os.path.exists(p):
        candidates.append(p)
if not candidates:
    candidates = sorted(z for z in glob.glob('/content/*.zip') if 'checkmaize' not in z)
if not candidates:
    print('No dataset zip found in /content. Upload it (Kaggle: crop-pest-and-disease-detection.zip,')
    print('or Mendeley: Raw Data.zip) using the files pane, then re-run this cell.')
    raise SystemExit
zip_path = candidates[0]
print('Using:', os.path.basename(zip_path))

extract_dir = '/content/ccmt_extract'
shutil.rmtree(extract_dir, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

dst = 'data/raw/ccmt_ghana'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)
copied = {}

def copy_dir(src, key):
    target = os.path.join(dst, NEEDED[key])
    shutil.copytree(src, target)
    copied[key] = len(os.listdir(target))

nested_maize = None
for root, dirs, files in os.walk(extract_dir):
    if 'Maize' in dirs:
        nested_maize = os.path.join(root, 'Maize')
        break

if nested_maize and any(k.lower() in NEEDED_LOWER for k in os.listdir(nested_maize)):
    for sub in sorted(os.listdir(nested_maize)):
        if sub.lower() in NEEDED_LOWER:
            copy_dir(os.path.join(nested_maize, sub), NEEDED_LOWER[sub.lower()])
            print(sub, copied[NEEDED_LOWER[sub.lower()]])
else:
    for folder in sorted(os.listdir(extract_dir)):
        full = os.path.join(extract_dir, folder)
        if folder.startswith('Maize ') and os.path.isdir(full):
            key = folder[len('Maize '):].lower()
            if key in NEEDED_LOWER:
                copy_dir(full, NEEDED_LOWER[key])
                print(key, copied[NEEDED_LOWER[key]])

missing = [k for k in NEEDED if k not in copied]
assert not missing, f'Could not find classes {missing} in {os.path.basename(zip_path)}'
print('ccmt_ghana ready:', copied)


In [ ]:
os.chdir('/content/checkmaize')
!python -m data.make_manifest
!python -m data.make_splits
!python -m pytest data/tests -v


In [ ]:
import shutil, os
shutil.make_archive('/content/splits', 'zip', 'data/manifests')
from google.colab import files
files.download('/content/splits.zip')
